<a href="https://colab.research.google.com/github/Gilleswint-s1128612/bachelor-thesis-LambdaMART-TAS-B/blob/main/Thesis_TAS_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
project_root = "/content/drive/MyDrive/TAS-B_MSMARCO_thesis"


In [ ]:
import os
# set Java 21
!apt-get update && apt-get install openjdk-21-jre-headless -qq > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java
!java -version

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,085 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [ ]:
!pip install pyserini
!pip install sentence_transformers
!pip install datasets
!pip install codecarbon
!pip install faiss-gpu-cu12


In [ ]:
# We reuse the bm25 docs already filtered in the LambdaMART part of the thesis
# contain pid and text
top1000_docs= "/content/drive/MyDrive/LamdaMART_MSMARCO_thesis/collections/msmarco-passage/filtered_collection.tsv"
# contain qid, pid and rank
bm25_run_log = "/content/drive/MyDrive/LamdaMART_MSMARCO_thesis/runs/run.dl19.bm25.tsv"

**TAS-B Passage Embedding**

In [ ]:
# embedding and reranking must be run after each other because the embeddings are stored in a dictionary
import os
import time
import torch
import numpy as np
import pandas as pd
from codecarbon import EmissionsTracker
from pyserini.search import get_topics
from sentence_transformers import SentenceTransformer, util
from tqdm.notebook import tqdm

%cd $project_root
# create directories for runs
os.makedirs("runs", exist_ok=True)

output_run_path = "runs/run.dl19.dense.tasb.txt"

idx_tracker = EmissionsTracker(
    project_name="3_TASB_Indexing",
    output_file="emissions.csv",
    measure_power_secs=5
)

# see if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
# implement TAS-B
model = SentenceTransformer('sebastian-hofstaetter/distilbert-dot-tas_b-b256-msmarco', device=device)

idx_tracker.start()
# creates a dictionary of passages and their text
pid_to_text_dict = {}
# creates a dictionary of qid and pid
bm25_dict = {}

try:
    # only embed the top 1000 selected by bm25
    with open(top1000_docs, "r", encoding="utf-8", errors="ignore") as f:

      for line in f:
          parts = line.strip().split("\t", 1)
          # check if it contains the pid and text part
          if len(parts) == 2:

            pid_to_text_dict[parts[0].strip()] = parts[1].strip()

    with open(bm25_run_log, "r", encoding="utf-8") as run_file:
        for line in run_file:
            parts= line.strip().split("\t")
            # check if there is 3 args qid, pid and rank
            if len(parts) >= 2:
                qid= parts[0].strip()
                pid= parts[1].strip()
                # check if the qid is already added
                if qid not in bm25_dict:
                  bm25_dict[qid] = []
                # add the passage to the query
                bm25_dict[qid].append(pid)

    # pre index document embeddings
    print("Getting unique pids")
    unique_pids = list(pid_to_text_dict.keys())
    unique_passage_txt = [pid_to_text_dict[pid] for pid in unique_pids]

    print(f"Encoding {len(unique_pids)} passages")
    # convert every passage to embedding
    passage_embeddings_matrix = model.encode(unique_passage_txt, batch_size=256, convert_to_tensor=True, show_progress_bar=True)
    # link pid to its embedding
    passage_embedding_lookup = {pid: passage_embeddings_matrix[i] for i, pid in enumerate(unique_pids)}

finally:
    idx_tracker.stop()
    print("Indexing and embedding complete!")

[codecarbon WARNING @ 16:58:41] Multiple instances of codecarbon are allowed to run at the same time.


/content/drive/MyDrive/TAS-B_MSMARCO_thesis
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Getting unique pids
Encoding 42804 passages


Batches:   0%|          | 0/168 [00:00<?, ?it/s]

Indexing and embedding complete!


**TAS-B Query embedding and dot product score**

In [ ]:

import os
import time
import torch
import numpy as np
import pandas as pd
from codecarbon import EmissionsTracker
from pyserini.search import get_topics
from sentence_transformers import SentenceTransformer, util
from tqdm.notebook import tqdm

%cd $project_root
rerank_tracker = EmissionsTracker(
    project_name="4_TASB_Reranking",
    output_file="emissions.csv",
    measure_power_secs=5
)
rerank_tracker.start()

try:
    # qid and query text
    queries_dl19 = get_topics('dl19-passage')

    #qid and pid
    qid_pid_dict = tqdm(bm25_dict.items(), desc="Neural Reranking")

    skipped_pids= 0
    total_written_lines= 0


    with open(output_run_path, "w", encoding="utf-8") as output_file:
        for qid, pids in qid_pid_dict:
            query_dict = queries_dl19.get(qid)
            if not query_dict:
                try:
                    query_dict = queries_dl19.get(int(qid))
                except ValueError:
                    pass

            if not query_dict:
                continue
            # get query text for dl19
            query_text = query_dict['title']
            # gets only the passage embeddings
            valid_embeddings = [passage_embedding_lookup[pid] for pid in pids]

            if not valid_embeddings:
                continue
            # stack embeddings together in their original order, to remain index positions for pids
            current_passage_matrix = torch.stack(valid_embeddings)
            # embed the queries
            query_embedding = model.encode(query_text, convert_to_tensor=True, show_progress_bar=True)
            # compute dot product between query embeddings and passage
            scores = util.dot_score(query_embedding, current_passage_matrix)[0]
            # rank scores, contains rank and index linked to passage
            ranked_indices = torch.argsort(scores, descending=True).cpu().numpy()
            # write the results in the follwoing way Qid, Pid, Rank
            for rank_idx, local_idx in enumerate(ranked_indices, start=1):
                output_file.write(f"{qid}\t{pids[local_idx]}\t{rank_idx}\n")
                total_written_lines += 1
    print(f"Total lines written to run file: {total_written_lines}")

    print(f"Total pids skipped{skipped_pids}")

except Exception as e:
    print(f"ERROR: {e}")
finally:
    rerank_tracker.stop()
    print("TAS-B reranking complete")

Neural Reranking:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Total lines written to run file: 43000
Total pids skipped0
TAS-B reranking complete


**TAS-B results**

In [ ]:
%cd $project_root
!python3 -m pyserini.eval.convert_msmarco_run_to_trec_run \
  --input runs/run.dl19.dense.tasb.txt \
  --output runs/run.dl19.dense.tasb.trec

!python3 -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage runs/run.dl19.dense.tasb.trec

Done!
[0.004s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.004s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.007s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.007s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
ndcg_cut_10           	all	0.5532
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pyserini/eval/trec_eval.py", line 213, in <module>
    trec_eval(sys.argv)
  File "/usr/local/lib/python3.12/dist-

In [ ]:
%cd $project_root
import pandas as pd

try:
    df = pd.read_csv("emissions.csv")
    columns_to_show = ["project_name", "duration", "emissions", "cpu_power", "gpu_power", "ram_power", "energy_consumed"]
    available_columns = [col for col in columns_to_show if col in df.columns]
    display(df[available_columns])

except FileNotFoundError:
    print("Emissions.csv not found.")
    df = None

/content/drive/MyDrive/TAS-B_MSMARCO_thesis


,project_name,duration,emissions,cpu_power,gpu_power,ram_power,energy_consumed
0,3_TASB_Indexing,149.604787,0.001146,3.529713,66.979898,10.0,0.003281
1,4_TASB_Reranking,4.896780,0.000019,7.495784,25.176352,10.0,0.000056
